# AI usecase decision tree optimization

## Processing

### Process the answers

#### Group identical nodes

In [29]:
def group_identical_answer_nodes(tree):
    grouped_answers = {}
    replace_by = []
    to_modify = set()
    new_tree = []

    for item in tree:
        if item['type'] == 'answer':
            output = item['content']
            for source in item['source']:
                output += ", " + source

            if output not in grouped_answers:
                grouped_answers[output] = []
            grouped_answers[output].append(item)

    for sources, items in grouped_answers.items():
        first_item = items[0]
        deleted_nodes = [item['id'] for item in items[1:]]

        in_values = []
        for item in items:
            if 'in' in item:
                in_values.extend(item['in'] if isinstance(item['in'], list) else [item['in']])
        to_modify.update(in_values)

        first_item['in'] = in_values

        replace_by.append({
            "id": first_item['id'],
            "deleted_nodes": deleted_nodes
        })

        new_tree.append(first_item)

    # Add untouched nodes
    for item in tree:
        if item['type'] != 'answer':
            new_tree.append(item)

    return new_tree, replace_by, list(to_modify)


#### Update parent nodes

In [30]:
def update_question_nodes(tree, replace_by, to_modify):
    updated_tree = []

    for item in tree:
        if item['type'] == 'question':
            if item['id'] in to_modify:
                updated_item = item.copy()

                for mapping in replace_by:
                    if updated_item['yes'] in mapping['deleted_nodes']:
                        updated_item['yes'] = mapping['id']
                        break

                for mapping in replace_by:
                    if updated_item['no'] in mapping['deleted_nodes']:
                        updated_item['no'] = mapping['id']
                        break

                updated_tree.append(updated_item)
            else:
                updated_tree.append(item)
        else:
            updated_tree.append(item)

    return updated_tree


### Process the question nodes

#### Group identical nodes

In [31]:
def group_identical_question_nodes(tree, to_modify):
    grouped_questions = {}
    question_replace_by = []
    question_to_modify = set()
    updated_tree = []
    processed = []

    # Collect question nodes from new_tree
    question_nodes = [node for node in tree if node['type'] == 'question' and node['id'] in to_modify]

    # Group question nodes by (content, yes, no)
    for item in question_nodes:
        key = (item['content'], item['yes'], item['no'])
        if key not in grouped_questions:
            grouped_questions[key] = []
        grouped_questions[key].append(item)

    # Process groups
    for key, items in grouped_questions.items():
        if len(items) > 1:
            first_item = items[0]
            deleted_nodes = [item['id'] for item in items[1:]]

            # Merge 'in' values
            in_values = []
            for item in items:
                if 'in' in item:
                    in_values.extend(item['in'] if isinstance(item['in'], list) else [item['in']])

            question_to_modify.update(in_values)

            # Update 'in' of surviving node
            first_item['in'] = in_values
            
            # Add to question_replace_by mapping
            question_replace_by.append({
                "id": first_item['id'],
                "deleted_nodes": deleted_nodes
            })

            updated_tree.append(first_item)
        # else:
        #     updated_tree.append(items[0])

    # # Add untouched question nodes
    # for item in tree:
    #     if item['type'] == 'question' and item['id'] not in [q['id'] for group in grouped_questions.values() for q in group]:
    #         updated_tree.append(item)

    # # Add answer nodes
    # for item in tree:
    #     if item['type'] == 'answer':
    #         updated_tree.append(item)

    # processed = []
    for q in question_replace_by:
        processed.append(q['id'])
        processed.extend(q['deleted_nodes'])

    return updated_tree, question_replace_by, list(question_to_modify), processed


#### Update parent nodes

In [32]:
def update_remaining_question_nodes(tree, new_tree, question_replace_by, question_to_modify, processed):
    updated_tree = new_tree

    for item in tree:

        # Check if this question node's id is in question_to_modify
        if item['id'] in question_to_modify:
            updated_item = item.copy()

            # Update 'yes' value if needed
            for mapping in question_replace_by:
                if updated_item['yes'] in mapping['deleted_nodes']:
                    updated_item['yes'] = mapping['id']
                    break

            # Update 'no' value if needed
            for mapping in question_replace_by:
                if updated_item['no'] in mapping['deleted_nodes']:
                    updated_item['no'] = mapping['id']
                    break

            # Add updated question node
            updated_tree.append(updated_item)
        # Node is not affected, just copy
        elif item['id'] not in processed:
            updated_tree.append(item)

    return updated_tree

#### Remove trivial nodes

In [33]:
def remove_trivial_nodes(tree):
    replace_by = []
    deleted_nodes = set()

    # Step 1: Identify trivial nodes
    for node in tree:
        if node['type'] == 'question' and node['yes'] == node['no']:
            replace_by.append({
                "deleted_node": node['id'],
                "id_out": node['yes'],
                "id_in": node['in'] if isinstance(node['in'], list) else [node['in']]
            })
            deleted_nodes.add(node['id'])
            deleted_nodes.add(node['yes'])
            for id_in in replace_by[-1]['id_in']:
                deleted_nodes.add(id_in)

    # Step 2: Prepare new tree (exclude deleted nodes)
    new_tree = [node for node in tree if node['id'] not in deleted_nodes]

    # Convert tree to dict for easy lookup
    tree_dict = {node['id']: node for node in tree}

    # Step 3: Apply replacements
    for mapping in replace_by:
        deleted_id = mapping['deleted_node']
        id_out = mapping['id_out']
        id_in = mapping['id_in']

        # Update node with id = id_out (replace 'in' values)
        if id_out in tree_dict:
            out_node = tree_dict[id_out].copy()
            # Merge 'in' values
            existing_in = out_node['in'] if isinstance(out_node['in'], list) else [out_node['in']]
            merged_in = list(set(existing_in + id_in))
            merged_in.remove(deleted_id)
            out_node['in'] = merged_in
            # Add updated node to new_tree
            # Avoid duplicates
            # new_tree = [n for n in new_tree if n['id'] != out_node['id']]
            new_tree.append(out_node)

        # Update node with id = id_in (swap yes/no if needed)
        for in_id in id_in:
            if in_id in tree_dict and tree_dict[in_id]['yes'] != tree_dict[in_id]['no']:
                in_node = tree_dict[in_id].copy()
                changed = False
                if in_node['yes'] == deleted_id:
                    in_node['yes'] = id_out
                    changed = True
                if in_node['no'] == deleted_id:
                    in_node['no'] = id_out
                    changed = True
                if changed:
                    # Replace in new_tree
                    # new_tree = [n for n in new_tree if n['id'] != in_node['id']]
                    new_tree.append(in_node)

    return new_tree


### Iteration over the tree

In [34]:
def reduce_tree(tree,remove_trivial):
    print("Starting reduction...")    
    print("current node count: ", len(tree))

    # Step 1: Merge identical answers and update parents
    new_tree, replace_by, to_modify = group_identical_answer_nodes(tree)
    new_tree = update_question_nodes(new_tree, replace_by, to_modify)

    # Save original tree
    tree = new_tree

    while True:
        print("current node count: ", len(tree))

        # Step 2: Merge identical questions layer by layer
        new_tree, question_replace_by, question_to_modify, processed = group_identical_question_nodes(new_tree, to_modify)
        new_tree = update_remaining_question_nodes(tree, new_tree, question_replace_by, question_to_modify, processed)

        print("current node count: ", len(new_tree))

        # Step 3: Remove trivial nodes
        if remove_trivial:
            new_tree = remove_trivial_nodes(new_tree)

        if len(new_tree) == len(tree):
            break  # No further reduction
        else:
            # Move to next row
            to_modify = question_to_modify 
            tree = new_tree

    print("Reduction complete.")
    return new_tree


## Implementation

### Load the data

In [42]:
import json

# Load the PDF paths from the JSON file
with open('data/sources/tree.json', 'r', encoding="UTF-8") as file:
    tree = json.load(file)

### Process the data

#### Keep trivial nodes

In [36]:
final_tree = reduce_tree(tree, False)

Starting reduction...
current node count:  95
current node count:  65
current node count:  61
current node count:  61
current node count:  61
Reduction complete.


#### Remove trivial nodes

In [43]:
final_tree_slim = reduce_tree(tree, True)
final_tree_slim

Starting reduction...
current node count:  95
current node count:  65
current node count:  61
current node count:  59
current node count:  59
Reduction complete.


[{'id': 'e18',
  'type': 'question',
  'content': 'Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?',
  'in': ['c18', 'e26'],
  'yes': 'g18',
  'no': 'g19'},
 {'id': 'k16',
  'type': 'question',
  'content': "Changement de palier selon l'ancien système?",
  'in': ['h15', 'k47'],
  'yes': 't16',
  'no': 't17'},
 {'id': 'r44',
  'type': 'question',
  'content': 'Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?',
  'in': ['p41', 'p36', 'k30'],
  'yes': 't45',
  'no': 'j21'},
 {'id': 'g19',
  'type': 'answer',
  'content': 'Rente linéaire',
  'source': ['Ch. 9102 CIRAI',
   'Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)'],
  'in': ['e18', 'h26']},
 {'id': 'g18',
  'type': 'answer',
  'content': 'Rente linéaire',
  'source': ['Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)'],
  'in': ['e18', 'h20', 'h26']},
 {'id': 'm25',
  'type': 'answer',
  'content': 'Rente linéaire',
  'sour

### Save the new trees

In [38]:
# Save the new_tree structure to a JSON file
with open('data/output/final_tree.json', 'w') as json_file:
    json.dump(final_tree, json_file, ensure_ascii=False)

print(f"Data saved to data/output/final_tree.json")

Data saved to data/output/final_tree.json


In [44]:
# Save the new_tree structure to a JSON file
with open('data/output/final_tree_slim.json', 'w', encoding="UTF-8") as json_file:
    json.dump(final_tree_slim, json_file, ensure_ascii=False)

print(f"Data saved to data/output/final_tree_slim.json")

Data saved to data/output/final_tree_slim.json


## Exploration

### Load the trees

In [47]:
# with open('data/output/final_tree_slim.json', 'r') as file:
#     optimized_tree2 = json.load(file)

# with open('data/output/final_tree.json', 'r') as file:
#     optimized_tree = json.load(file)

# with open('data/sources/tree.json', 'r') as file:
#     original_tree = json.load(file)

# with open('data/sources/tree2.json', 'r') as file:
#     tree2 = json.load(file)

with open('data/output/final_tree_slim.json', 'r', encoding="UTF-8") as file:
    tree = json.load(file)

In [48]:
# Preprocess the tree: {id: node}
# opti_dict = {node['id']: node for node in optimized_tree}
# og_dict = {node['id']: node for node in original_tree}
# t2_dict = {node['id']: node for node in tree2}
t_dict = {node['id']: node for node in tree}
t_dict

{'e18': {'id': 'e18',
  'type': 'question',
  'content': 'Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?',
  'in': ['c18', 'e26'],
  'yes': 'g18',
  'no': 'g19'},
 'k16': {'id': 'k16',
  'type': 'question',
  'content': "Changement de palier selon l'ancien système?",
  'in': ['h15', 'k47'],
  'yes': 't16',
  'no': 't17'},
 'r44': {'id': 'r44',
  'type': 'question',
  'content': 'Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?',
  'in': ['p41', 'p36', 'k30'],
  'yes': 't45',
  'no': 'j21'},
 'g19': {'id': 'g19',
  'type': 'answer',
  'content': 'Rente linéaire',
  'source': ['Ch. 9102 CIRAI',
   'Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)'],
  'in': ['e18', 'h26']},
 'g18': {'id': 'g18',
  'type': 'answer',
  'content': 'Rente linéaire',
  'source': ['Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)'],
  'in': ['e18', 'h20', 'h26']},
 'm25': {'id': 'm25',
  'type': 'answe

### Find the root

In [49]:
root = tree[0]
for node in tree:
    if node['in'] == [] or node['id'] == 0:
        root = node
        break
root['id']

'b26'

### Exploring functions

#### Explore a specific path

In [50]:
def explore_custom_path(start_id, tree_dict, path_choices):
    current_id = start_id
    path = []
    choice_index = 0

    while True:
        node = tree_dict.get(current_id)

        if not node:
            print(f"Node with id {current_id} not found!")
            break

        path.append(node)

        if node['type'] == 'answer':
            print(f"Reached answer node: {node['id']}")
            break

        # Determine which branch to follow
        if choice_index >= len(path_choices):
            print(f"No more choices left, stopping at node {node['id']}")
            break

        choice = path_choices[choice_index]
        if choice not in ["yes", "no"]:
            print(f"Invalid choice '{choice}' at step {choice_index}, stopping.")
            break

        current_id = node[choice]
        choice_index += 1

    return path


#### Explore all possible paths

In [51]:
def explore_all_paths(start_id, tree_dict):
    all_paths = []

    def dfs(current_id, current_path):
        node = tree_dict.get(current_id)
        if not node:
            print(f"Node with id {current_id} not found!")
            return

        # Add current node to the path
        current_path.append(node)

        # If answer node, record path
        if node['type'] == 'answer':
            all_paths.append(list(current_path))  # Copy of current_path
        else:
            # Recurse on "yes" branch
            dfs(node['yes'], list(current_path))  # Copy to avoid mutation

            # Recurse on "no" branch
            dfs(node['no'], list(current_path))

    dfs(start_id, [])
    return all_paths


#### Display a path

In [52]:
def display_path(path):
    print("Explored Path:")
    for node in path:
        if node['type'] == 'question':
            print(f"Node ID: {node['id']}, Content: {node['content']}")
        if node['type'] == 'answer':
            print(f"Node ID: {node['id']}, Content: {node['content']}, Sources: {node['source']}")

#### Save a path

In [64]:
def save_paths_by_node_id(all_paths, file_name='all_paths_by_id'):
    output_path = f"data/output/{file_name}.json"
    all_paths_by_id = []

    for path in all_paths:

        reduced_path = {
            'path': [{"question": node['content'], "answer": 'Oui' if path[i+1]['id'] == node['yes'] else 'Non'} for i, node in enumerate(path[:-1])],
            'answer': {}
        }

        # Get the final answer node
        answer_node = path[-1]
        if answer_node['type'] == 'answer':
            reduced_path['answer'] = {"decision": answer_node['content'], "sources": answer_node['source']}

        all_paths_by_id.append(reduced_path)

    # Save to JSON file
    with open(output_path, 'w', encoding="UTF-8") as json_file:
        json.dump(all_paths_by_id, json_file, ensure_ascii=False, indent=2)

    return all_paths_by_id

### Explore the optimized tree

#### Specific path

In [ ]:
custom_path = ["yes", "no", "yes", "yes"]
opti_test_path = explore_custom_path(root['id'], opti_dict, custom_path)

display_path(opti_test_path)

Reached answer node: g2
Explored Path:
Node ID: b26, Content: S'agit-il d'une révision sur demande ou d'une révision d'office?
Node ID: c26, Content: Droit ouvert dans le système linéaire ?
Node ID: e34, Content: Y a-t-il eu une modification des faits entre le 01.01.2022 et le 31.12.2023 ?
Node ID: h34, Content: L'âge de l’assuré, au 01.01.2022, est-il égal ou supérieur (=>) à 55 ans ?
Node ID: g2, Content: Rente par pallier, Sources: ['Lettre c des dispositions transitoires de la modification du 19 juin 2020 (Développement continu de l’AI)']


#### All paths

In [56]:
opti_all_paths = explore_all_paths(root['id'], t_dict)

print(len(opti_all_paths), "Possible Paths")

46 Possible Paths


In [65]:
all_paths = save_paths_by_node_id(opti_all_paths, "all_paths_tree")

In [ ]:
answers = [
    "Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ? - Non",
    "Changement de palier selon l'ancien système? - Non",
    "S'agit-il d'une révision sur demande ou d'une révision d'office? - Oui",
    "S'agit-il d'une 1ère demande RER ou demande subséquente dépôsée avant le 01.07.2021 et échéance délai de carence avant le 01.01.2022? - Non",
    "Y a-t-il eu une modification des faits entre le 01.01.2022 et le 31.12.2023 ? - Non",
    "Le degré d’invalidité s’est-il modifié d’au-moins 5% ? - Non",
    "Le degré d’invalidité est-il augmenté ? - Non",
    "L'âge de l’assuré, au 01.01.2022, est-il égal ou supérieur (=>) à 55 ans ? - Non",
    "Le taux d'invalidité est-il d'au-moins 50% ? - Non",
    "Le montant de la rente est-il diminué ? - Non",
    "Le taux d'invalidité est-il d'au-moins 70% ? - Non",
    "Droit ouvert dans le système linéaire ? - Non",
    "Le montant de la rente est-il augmenté ? - Non"
]

check = 0
for path in all_paths:
    dif = False
    for qa in path['path']:
        if qa not in answers:
            # print(f"qa: {qa}, not in answers: {answers}")
            dif = True
            break
    if not dif:
        print(path)
        check += 1
        break
if check == 0:
    print("false")

{'path': ["S'agit-il d'une révision sur demande ou d'une révision d'office? - Oui", 'Droit ouvert dans le système linéaire ? - Non', 'Y a-t-il eu une modification des faits entre le 01.01.2022 et le 31.12.2023 ? - Non', 'Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ? - Non'], 'answer': "Décision: Rente par pallier, Sources: ['Art 17 al. 1 let. b LPGA', 'Art. 87 RAI', 'Ch. 5101 CIRAI', 'Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)']"}


##### Check for identical paths

In [ ]:
all_paths_content = []
all_paths_content_sorted = []
for path in opti_all_paths:
    all_paths_content_sorted.append(sorted([p['content'] if p['type'] == 'question' else f"{p['content']}, {p['source']}" for p in path]))
    all_paths_content.append([p['content'] if p['type'] == 'question' else f"{p['content']}, {p['source']}" for p in path])

len(all_paths_content)

48

In [ ]:
checked_path = []
doubled_path = []
for i, path in enumerate(all_paths_content_sorted):
    if path not in checked_path:
        checked_path.append(path)
    else:
        doubled_path.append(all_paths_content[i])

len(doubled_path)

5

##### Save all possible paths in a JSON file

In [ ]:
all_paths_reduced = []

for path in opti_all_paths:
    reduced_path = {'path': [], 'answer': ''}
    for i, node in enumerate(path):
        if node['type'] == 'question':
            if node['yes'] == path[i+1]['id']:
                reduced_path['path'].append(f"{node['content']} - yes")
            else:
                reduced_path['path'].append(f"{node['content']} no")
        else:
            reduced_path['answer'] = f"Décision: {node['content']}, Sources: {node['source']}"
    all_paths_reduced.append(reduced_path)

# Save the new_tree structure to a JSON file
with open('data/output/all_paths.json', 'w') as json_file:
    json.dump(all_paths_reduced, json_file, ensure_ascii=False)

print(f"Data saved to data/output/all_paths.json")

Data saved to data/output/all_paths.json


### Explore the original tree

#### All paths

In [ ]:
og_all_paths = explore_all_paths(root['id'], og_dict)

print(len(og_all_paths), "Possible Paths")

48 Possible Paths


### Explore tree2

#### All paths

In [ ]:
tree2_all_paths = explore_all_paths('0', t2_dict)

print(len(tree2_all_paths), "Possible Paths")

18 Possible Paths


In [ ]:
save_paths_by_node_id(tree2_all_paths, "all_paths_tree2")

Data saved to data/output/all_paths_tree2.json


### Compare outcomes

In [ ]:
similarity_ratio = 0

for i, path in enumerate(og_all_paths):
    print(f"Path {i}:")

    print(f"\nOriginal")
    display_path(path)

    print(f"\nOptimized")
    display_path(opti_all_paths[i])
    print(path[-1])

    og_outcome = f"{path[-1]['content']}, source: {path[-1]['source']}"
    opti_outcome = f"{opti_all_paths[i][-1]['content']}, source: {opti_all_paths[i][-1]['source']}"

    if og_outcome == opti_outcome:
        print(f"\nSame outcome:")
        print(f"Outcome: {og_outcome}")
        similarity_ratio += 1
        
    else:
        print(f"\nDifferent outcomes:")
        print(f"Optimized outcome: {opti_outcome}")
        print(f"Original outcome: {og_outcome}")

    print("\n----------------------------------------\n")

print(f"Similarity Ratio: {similarity_ratio / len(og_all_paths) * 100}%")

Path 0:

Original
Explored Path:
Node ID: b26, Content: S'agit-il d'une révision sur demande ou d'une révision d'office?
Node ID: c26, Content: Droit ouvert dans le système linéaire ?
Node ID: e26, Content: Y a-t-il eu une modification des faits entre le 01.01.2022 et le 31.12.2023 ?
Node ID: h28, Content: Le degré d’invalidité s’est-il modifié d’au-moins 5% ?
Node ID: k30, Content: Le degré d’invalidité est-il augmenté ?
Node ID: n30, Content: Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?
Node ID: t30, Content: Rente linéaire, Sources: ['Lettre b points 1 et 2 des dispositions transitoires de la modification du 19 juin 2020 (Développement continu de l’AI)', 'Lettre 1 des dispositions transitoires relatives à la modification du 18 octobre 2023 (RAI)']

Optimized
Explored Path:
Node ID: b26, Content: S'agit-il d'une révision sur demande ou d'une révision d'office?
Node ID: c26, Content: Droit ouvert dans le système linéaire ?
Node ID: e26, Content: Y a-t-il eu une modifica

In [ ]:
def check_path(path):
    found = False
    for p in all_paths_reduced:
        if path == p['path']:
            print(p['answer'])
            found = True
            break
    if not found:
        print("No matching path found.")

In [ ]:
check_path(["yes", "no", "yes", "yes"])

No matching path found.


# Comparaison vd et fb

In [ ]:
import json

with open('data/output/final_treev2.json', 'r') as file:
    tree2 = json.load(file)
    ### Load the trees
# with open('data/output/final_tree_slim.json', 'r') as file:
    # tree1 = json.load(file)

In [ ]:
answer_t1 = []
for t in tree2:
    if t['type'] == 'answer' and t['content'] not in answer_t1:
        answer_t1.append(f"Décision: {t['content']}, Sources: {t['source']}")
answer_t1

["Décision: Rente linÃ©aire, Sources: ['Ch. 9102 CIRAI', 'Lettre 1 des dispositions transitoires relatives Ã\\xa0 la modification du 18 octobre 2023 (RAI)']",
 "Décision: Rente linÃ©aire, Sources: ['Lettre 1 des dispositions transitoires relatives Ã\\xa0 la modification du 18 octobre 2023 (RAI)']",
 "Décision: Rente linÃ©aire, Sources: ['Art. 88bis al. 2 RAI', 'Art 17 al. 1 let. b LPGA', 'Lettre 1 des dispositions transitoires relatives Ã\\xa0 la modification du 18 octobre 2023 (RAI)']",
 "Décision: Rente par pallier, Sources: ['Lettre b points 1 et 2 des dispositions transitoires de la modification du 19 juin 2020 (DÃ©veloppement continu de lâ€™AI)', 'Lettre 1 des dispositions transitoires relatives Ã\\xa0 la modification du 18 octobre 2023 (RAI)']",
 "Décision: Rente par pallier, Sources: ['Art 17 al. 1 let. b LPGA', 'Art. 87 RAI', 'Ch. 5101 CIRAI']",
 "Décision: Rente par pallier, Sources: ['Lettre b point 1 des dispositions transitoires de la modification du 19 juin 2020 (DÃ©velopp

In [ ]:
questions_t1 = []
for t in tree2:
    if t['type'] == 'question' and t['content'] not in questions_t1:
        questions_t1.append(f"{t['content']}")
questions_t1


['Y a-t-il eu une augmentation du taux depuis le 01.01.2024 ?',
 "Changement de palier selon l'ancien systÃ¨me?",
 "S'agit-il d'une rÃ©vision sur demande ou d'une rÃ©vision d'office?",
 "S'agit-il d'une 1Ã¨re demande RER ou demande subsÃ©quente dÃ©pÃ´sÃ©e avant le 01.07.2021 et Ã©chÃ©ance dÃ©lai de carence avant le 01.01.2022?",
 'Y a-t-il eu une modification des faits entre le 01.01.2022 et le 31.12.2023 ?',
 'Le degrÃ© dâ€™invaliditÃ© sâ€™est-il modifiÃ© dâ€™au-moins 5% ?',
 'Le degrÃ© dâ€™invaliditÃ© est-il augmentÃ© ?',
 "L'Ã¢ge de lâ€™assurÃ©, au 01.01.2022, est-il Ã©gal ou supÃ©rieur (=>) Ã\xa0 55 ans ?",
 "Le taux d'invaliditÃ© est-il d'au-moins 50% ?",
 'Le montant de la rente est-il diminuÃ© ?',
 "Le taux d'invaliditÃ© est-il d'au-moins 70% ?",
 'Droit ouvert dans le systÃ¨me linÃ©aire ?',
 'Le montant de la rente est-il augmentÃ© ?']

In [ ]:
questions_t2 = []
for t in tree2:
    if t['type'] == 'question' and t['content'] not in questions_t2:
        questions_t2.append(t['content'])
questions_t2


['Début du versement de la rente après le 1er janvier 2022 ?',
 'Révision d’office au 01.01.2024, PA née après 1966 ?',
 'Présence d’un motif de révision ?',
 'PA née après 1966 ?',
 "Modification du taux d'invalidité d'au moins 5% par rapport au taux précédent ?",
 'Taux d’invalidité augmente ?',
 'Quotité / montant de la rente augmente?',
 'Changement de palier à la hausse ?']

In [66]:
import json

def escape_java_string(s):
    return s.replace('\\', '\\\\').replace('"', '\\"')

def json_to_java_string_literal(json_path, output_path=None):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = f.read()

    lines = data.splitlines()
    java_string = 'String json =\n'

    for i, line in enumerate(lines):
        escaped = escape_java_string(line)
        suffix = '" +\n' if i < len(lines) - 1 else '";\n'
        java_string += f'  "{escaped}{suffix}'

    if output_path:
        with open(output_path, 'w', encoding='utf-8') as out:
            out.write(java_string)
        print(f"Java string saved to: {output_path}")
    else:
        print(java_string)

json_to_java_string_literal('data/output/all_paths_tree.json', 'data/output/all_paths_tree_converted.java')

Java string saved to: data/output/all_paths_tree_converted.java
